# 03 — How does a reranker work? Two-stage retrieval

Companion notebook to blog post **03 (Reranking)**. No API key needed — every model runs
locally.

The senior-librarian metaphor:
1. The **junior helper sprints** — stage-1 retriever, fast cover-level judgment
2. The **librarian reads** — cross-encoder: question + document together, one score
3. She **never fetches** — rerankers only reorder the retrieved pool
4. She's **slow** — so she only reads the shortlist

Further reading is listed in the series' `resources.md`; here every step runs.

In [ ]:
%pip install -q fastembed numpy flashrank

## Corpus — every villain this series has collected

Handbook one-liners (posts 00/01) + the error-code twins (post 2b).

In [ ]:
corpus = [
    "Grooming appointments must be booked at least 48 hours in advance",           # 0
    "The boarding facility closes at 7 pm on weekdays and 5 pm on weekends",       # 1
    "Dogs staying longer than three nights receive a complimentary bath before pickup",  # 2
    "Refunds for cancelled boarding are issued within 5 business days",            # 3
    "Bookings made for public holidays are non-refundable",                        # 4
    "Refunds for cancelled grooming appointments are issued within 10 business days",    # 5
    "All pets must have up to date rabies vaccination records on file",            # 6
    "Daycare drop off starts at 6:30 am and the last pickup is at 8 pm",           # 7
    "A late pickup fee of 15 dollars applies for every 30 minutes after closing",  # 8
    "Error E-4042 refund transaction declined by the payment gateway",             # 9
    "Error E-4043 refund transaction succeeded but receipt email failed",          # 10
    "Error E-4044 refund transaction pending manual review",                       # 11
]

## Stage 1 — the junior helper (post 2b's bi-encoder, verbatim)

In [ ]:
import numpy as np
from fastembed import TextEmbedding

emb_model = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
doc_embs = list(emb_model.embed(corpus))          # precomputed ONCE — the bi-encoder superpower

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def vector_search(query, top_k=5):
    q = list(emb_model.embed([query]))[0]
    return sorted(((cosine(q, e), i) for i, e in enumerate(doc_embs)), reverse=True)[:top_k]

## Stage 2 — the librarian (cross-encoder)

One model reads question + document TOGETHER and outputs one relevance score per pair.
Nothing precomputable — the question is part of the input. Scores are raw logits
(can be negative); only the ORDER matters.

In [ ]:
from fastembed.rerank.cross_encoder import TextCrossEncoder

reranker = TextCrossEncoder(model_name="Xenova/ms-marco-MiniLM-L-6-v2")

def rerank(query, doc_ids):
    scores = list(reranker.rerank(query, [corpus[i] for i in doc_ids]))
    return sorted(zip(scores, doc_ids), reverse=True)

## Rescue 1 — the rule/exception flip (post 01's villain)

Bi-encoder ranks the RULE above the EXCEPTION (cover-level word overlap). The
cross-encoder reads each doc WITH the question and flips them — the exception answers
a holiday question.

In [ ]:
q = "I booked boarding for a public holiday and want to cancel. Will I get my money back?"

dense = vector_search(q, 5)
print("BEFORE — bi-encoder order:")
for s, i in dense:
    print(f"  {s:7.3f}  doc {i}: {corpus[i][:65]}")

print("\nAFTER — cross-encoder rerank of that pool:")
for s, i in rerank(q, [i for _, i in dense]):
    print(f"  {s:7.3f}  doc {i}: {corpus[i][:65]}")

## Rescue 2 — the identifier twins (post 2b's villain)

Vector search confused E-4042/E-4043 (compression). The cross-encoder reads raw text —
the literal token sits in its attention. Second cure for the same disease (post 2c's
hybrid was the first).

In [ ]:
q = "error E-4042"

dense = vector_search(q, 5)
print("BEFORE — bi-encoder order:")
for s, i in dense:
    print(f"  {s:7.3f}  doc {i}: {corpus[i][:65]}")

print("\nAFTER — cross-encoder rerank:")
for s, i in rerank(q, [i for _, i in dense]):
    print(f"  {s:7.3f}  doc {i}: {corpus[i][:65]}")

## The price tag — librarian rule 4

Bi-encoder cost barely grows with corpus size (vectors precomputed, ANN sub-linear).
Cross-encoder pays full model cost PER PAIR at query time. The ratio is why two-stage
retrieval exists.

In [ ]:
import time

q = "error E-4042"
t0 = time.perf_counter()
for _ in range(10):
    vector_search(q, 12)
t_bi = (time.perf_counter() - t0) / 10

t0 = time.perf_counter()
for _ in range(10):
    rerank(q, list(range(12)))
t_cross = (time.perf_counter() - t0) / 10

print(f"bi-encoder  (query vs 12 precomputed vectors): {t_bi*1000:6.1f} ms")
print(f"cross-encoder (12 full query+doc reads)      : {t_cross*1000:6.1f} ms   ({t_cross/t_bi:.0f}x slower)")

## The pool limit — what a reranker can NEVER fix

"Saturday" ≠ "weekends": the right doc sits at dense rank 5. Rerank the top-3 pool —
the librarian diligently re-sorts three wrong books. Widen to top-6 — the right doc
gets lifted but this small L-6 model still gets fooled by surface overlap (all scores
negative = not confident in anything). Two lessons: pool width is the hard ceiling,
and rerankers sharpen order, they don't create relevance.

In [ ]:
q = "Until what time can I pick up my dog on a Saturday?"

dense_all = vector_search(q, 12)
print("full bi-encoder ranking:")
for rank, (s, i) in enumerate(dense_all, 1):
    mark = "  <-- the right doc (weekends = Saturday)" if i == 1 else ""
    print(f"  rank {rank:<3}{s:7.3f}  doc {i}: {corpus[i][:55]}{mark}")

for k in (3, 6):
    pool = [i for _, i in dense_all[:k]]
    print(f"\nrerank of top-{k} pool:")
    for s, i in rerank(q, pool):
        print(f"  {s:7.3f}  doc {i}: {corpus[i][:55]}")

## PRODUCTION — FlashRank (what the course uses)

Cross-encoder on local ONNX: no GPU, no API, ~120 MB. Reports probabilities instead of
logits. The course wires it into LlamaIndex as a postprocessor: **retrieve 20 → rerank →
keep 4**. Measured at course scale: faithfulness 0.865→0.909 (best), recall 0.73→0.73
(the pool limit!), latency 1.36→2.56 s (the librarian's fee).

In [ ]:
from flashrank import Ranker, RerankRequest

ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2")

q = "I booked boarding for a public holiday and want to cancel. Will I get my money back?"
pool_ids = [i for _, i in vector_search(q, 5)]
passages = [{"id": i, "text": corpus[i]} for i in pool_ids]

for p in ranker.rerank(RerankRequest(query=q, passages=passages)):
    print(f"  {p['score']:.4f}  doc {p['id']}: {p['text'][:65]}")

## Recap — the librarian's rules

1. **Junior sprints** → bi-encoder retrieval: precomputed, fast, cover-level
2. **Librarian reads** → cross-encoder: question + doc together; caught the exception
   and the exact token that separate encodings missed
3. **Never fetches** → only reorders the pool; recall can't rise (0.73 → 0.73)
4. **Slow, so shortlist** → ~8× per-doc cost: retrieve wide (20–100), rerank, keep 3–5

**Next: query transforms (04)** — attack the pool itself: rewrite the question so the
right documents get fetched in the first place.